In [12]:
# W tym pliku obliczone zostaną średnie histogramy, wartosci i odchylenia standardowe dla autentycznych twarzy
# Następnie przeprowadzimy analizę statystyczną dla przykładowych twarzy - autentycznej i podstawionej
import cv2
import cv2.data
import numpy as np
import matplotlib.pyplot as plt
import os

# Wczytanie zbioru autentycznych twarzy - losowe 500 obrazów
seed = 61185
np.random.seed(seed)
image_path = "../../../Dane/Humans"
image_count = 500

files = [f for f in os.listdir(image_path) if f.endswith(".jpg") or f.endswith(".jpeg") or f.endswith(".png") or f.endswith(".webp")] # Tylko pliki JPG i PNG i WEBP

# Wczytanie obrazów
images = []

for i in range(image_count):
    # Wybór losowych 500 plików
    file = np.random.choice(files)
    files.remove(file)
    image = cv2.imread(str(os.path.join(image_path, file)))
    images.append(image)
    
    
print(len(images))

In [13]:
# Wykrywanie twarzy
face_classifier = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def detect_face(image):
    faces = face_classifier.detectMultiScale(image, 1.1, 5)
    if len(faces) == 0:
        return None
    return faces[0]

In [14]:
# Obliczenie kontekstu otoczenia twarzy
def get_face_context(img, face_coordinates):
    # 1. Obliczenie proporcji twarzy w stosunku do całego obrazu
    # 2. Obliczenie współrzędnych punktów w bezpośrednim otoczeniu twarzy
    # 3. Obliczenie współrzędnych punktów w dalszym otoczeniu twarzy
    # 4. Zwrócenie współrzędnych z funkcji
    x, y, w, h = face_coordinates
    img_h, img_w, _ = img.shape

    # Obliczenie proporcji twarzy w stosunku do całego obrazu
    face_proportion = (w * h) / (img_h * img_w)

    # Uznajemy, że bezpośrednie otoczenie twarzy to 40% wysokości i szerokości twarzy
    # [PARAMETR]
    context_proportion = 0.40

    # Obliczenie współrzędnych punktów w bezpośrednim otoczeniu twarzy
    rectangle_point = (x - (context_proportion * w), y - (context_proportion * h))
    rectangle_width = w + 2 * (context_proportion * w)
    rectangle_height = h + 2 * (context_proportion * h)

    # Obliczenie współrzędnych punktów w dalszym otoczeniu twarzy
    rectangle_point_far = (x - (2 * context_proportion * w), y - (2 * context_proportion * h))
    rectangle_width_far = w + 4 * (context_proportion * w)
    rectangle_height_far = h + 4 * (context_proportion * h)

    # Sprawdzenie, czy punkty nie wychodzą poza obraz
    if rectangle_point[0] < 0:
        rectangle_point = (0, rectangle_point[1])
    if rectangle_point[1] < 0:
        rectangle_point = (rectangle_point[0], 0)
    if rectangle_point[0] + rectangle_width > img_w:
        rectangle_width = img_w - rectangle_point[0]
    if rectangle_point[1] + rectangle_height > img_h:
        rectangle_height = img_h - rectangle_point[1]

    if rectangle_point_far[0] < 0:
        rectangle_point_far = (0, rectangle_point_far[1])
    if rectangle_point_far[1] < 0:
        rectangle_point_far = (rectangle_point_far[0], 0)
    if rectangle_point_far[0] + rectangle_width_far > img_w:
        rectangle_width_far = img_w - rectangle_point_far[0]
    if rectangle_point_far[1] + rectangle_height_far > img_h:
        rectangle_height_far = img_h - rectangle_point_far[1]

    # Zwrócenie wartości
    return {
        'face_proportion': face_proportion,
        'rectangle_point': rectangle_point,
        'rectangle_width': rectangle_width,
        'rectangle_height': rectangle_height,
        'rectangle_point_far': rectangle_point_far,
        'rectangle_width_far': rectangle_width_far,
        'rectangle_height_far': rectangle_height_far
    }

In [15]:
# Obliczenie pojedynczego i średniego histogramu
def analyze_edge_distribution(sobel):
    edge_data = sobel.flatten()

    # Utworzenie histogramu na podstawie intensywności krawędzi, podzielonego na 16 przedziałów
    # [PARAMETR]
    hist, bins = np.histogram(edge_data, bins=16)

    return hist, bins


def average_hist(histograms):
    hist_array = np.array(histograms)

    average_hist = np.mean(hist_array, axis=0)

    return average_hist

In [16]:
# Obliczenie intensywności dla pojedynczego obrazu
# Sprawdzany będzie tylko region wokół twarzy o podanej szerokości (w) i wysokości (h)
def calculate_edge_intensity(img_sobel, face_topleft, face_botright):
    # Uzyskanie regionu bez samej twarzy
    x1, y1 = face_topleft
    x2, y2 = face_botright
    # 1. [:y1, :] - górna część obrazu
    # 2. [y2:, :] - dolna część obrazu
    # 3. [y1:y2, :x1] - lewa część obrazu bez powtórzeń
    # 4. [y1:y2, x2:] - prawa część obrazu bez powtórzeń
    top = img_sobel[:y1, :]
    bottom = img_sobel[y2:, :]
    left = img_sobel[y1:y2, :x1]
    right = img_sobel[y1:y2, x2:]

    # Obliczenie intensywności krawędzi dla wszystkich regionów
    top_sum = np.sum(top)
    bottom_sum = np.sum(bottom)
    left_sum = np.sum(left)
    right_sum = np.sum(right)

    top_count = np.count_nonzero(top)
    bottom_count = np.count_nonzero(bottom)
    left_count = np.count_nonzero(left)
    right_count = np.count_nonzero(right)

    # Obliczenie średniej intensywności krawędzi
    avg_intensity = (top_sum + bottom_sum + left_sum + right_sum) / (top_count + bottom_count + left_count + right_count)

    return avg_intensity

In [17]:
# Przeprowadzenie analizy dla każdego zdjęcia po kolei
histograms = []

for img in images:
    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    face = detect_face(img)
    
    if face is None:
        continue
    
    face_context = get_face_context(img, face)

    x, y, w, h = face

    # Póki co tylko Sobel X dla dalszej okolicy
    rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
    rectangle_width_far = int(face_context['rectangle_width_far'])
    rectangle_height_far = int(face_context['rectangle_height_far'])

    context_area_far = img[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]
    context_area_far_gray = cv2.cvtColor(context_area_far, cv2.COLOR_RGB2GRAY)

    context_area_edges_y = cv2.Sobel(context_area_far_gray, cv2.CV_64F, 0, 1, ksize=3)
    context_area_edges_y = cv2.convertScaleAbs(context_area_edges_y)

    hist, bins = analyze_edge_distribution(context_area_edges_y)
    histograms.append(hist)
    
average_histogram = average_hist(histograms)

# Obliczenie średniej i odchylenia standardowego dla autentycznych twarzy
# Dla każdego przedziału z osobna
hist_mean = np.mean(histograms, axis=0)
hist_std = np.std(histograms, axis=0)

print("Średnia histogramu krawędzi dla autentycznych twarzy:")
print(hist_mean)
print("Odchylenie standardowe histogramu krawędzi dla autentycznych twarzy:")
print(hist_std)

plt.bar(bins[:-1], average_histogram, width=(bins[1] - bins[0]), color='b', alpha=0.7)
plt.title("'Średni' histogram krawędzi dla autentycznych twarzy, dalszy kontekst, sobel Y")
plt.show()

# Zapisanie wartości do pliku
np.savez("histogram_data_far_Y.npz", average_histogram=average_histogram, bins=bins, hist_mean=hist_mean, hist_std=hist_std)

In [18]:
# Sprawdzenie dla przykładowych twarzy - autentycznej i podstawionej
# Wczytanie przykładowych twarzy
auth_face_path = "../../../Dane/Sample/Default/1 (34).jpeg"
auth_face = cv2.imread(auth_face_path)
auth_face = cv2.resize(auth_face, (256, 256))

face = detect_face(auth_face)
face_context = get_face_context(auth_face, face)

rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
rectangle_width_far = int(face_context['rectangle_width_far'])
rectangle_height_far = int(face_context['rectangle_height_far'])

context_area_far = auth_face[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]
context_area_far_gray = cv2.cvtColor(context_area_far, cv2.COLOR_RGB2GRAY)

context_area_edges_y = cv2.Sobel(context_area_far_gray, cv2.CV_64F, 0, 1, ksize=3)
context_area_edges_y = cv2.convertScaleAbs(context_area_edges_y)

hist, bins = analyze_edge_distribution(context_area_edges_y)

# Obliczenie z-Score
z_score = (hist - hist_mean) / hist_std

print("Z-Score dla przykładowej twarzy autentycznej:")
print(z_score)

plt.imshow(context_area_edges_y, cmap='gray')
plt.title("Krawędzie dla przykładowej twarzy autentycznej")
plt.axis('off')
plt.show()

In [19]:
spoof_face_path = "../../../Dane/Sample/zd/0.webp"
spoof_face = cv2.imread(spoof_face_path)
spoof_face = cv2.resize(spoof_face, (256, 256))

face = detect_face(spoof_face)
face_context = get_face_context(spoof_face, face)

rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
rectangle_width_far = int(face_context['rectangle_width_far'])
rectangle_height_far = int(face_context['rectangle_height_far'])

context_area_far = spoof_face[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]
context_area_far_gray = cv2.cvtColor(context_area_far, cv2.COLOR_RGB2GRAY)

context_area_edges_y = cv2.Sobel(context_area_far_gray, cv2.CV_64F, 0, 1, ksize=3)
context_area_edges_y = cv2.convertScaleAbs(context_area_edges_y)

hist, bins = analyze_edge_distribution(context_area_edges_y)

# Obliczenie z-Score
z_score = (hist - hist_mean) / hist_std

print("Z-Score dla przykładowej twarzy podstawionej:")
print(z_score)

plt.imshow(context_area_edges_y, cmap='gray')
plt.title("Krawędzie dla przykładowej twarzy podstawionej")
plt.axis('off')
plt.show()

In [20]:
# Histogram dla twarzy podstawionej
plt.bar(bins[:-1], hist, width=(bins[1] - bins[0]), color='r', alpha=0.7)
plt.title("Histogram krawędzi dla przykładowej twarzy podstawionej, dalszy kontekst, sobel Y")
plt.show()

In [21]:
# Przeprowadzenie analizy dla każdego zdjęcia po kolei
# Średni histogram dla osi Y
histograms = []

for img in images:
    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    face = detect_face(img)

    if face is None:
        continue

    face_context = get_face_context(img, face)

    x, y, w, h = face

    # Póki co tylko Sobel X dla dalszej okolicy
    rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
    rectangle_width_far = int(face_context['rectangle_width_far'])
    rectangle_height_far = int(face_context['rectangle_height_far'])

    context_area_far = img[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]
    context_area_far_gray = cv2.cvtColor(context_area_far, cv2.COLOR_RGB2GRAY)

    context_area_edges_x = cv2.Sobel(context_area_far_gray, cv2.CV_64F, 1, 0, ksize=3)
    context_area_edges_x = cv2.convertScaleAbs(context_area_edges_x)

    hist, bins = analyze_edge_distribution(context_area_edges_x)
    histograms.append(hist)

average_histogram = average_hist(histograms)

# Obliczenie średniej i odchylenia standardowego dla autentycznych twarzy
# Dla każdego przedziału z osobna
hist_mean = np.mean(histograms, axis=0)
hist_std = np.std(histograms, axis=0)

print("Średnia histogramu krawędzi dla autentycznych twarzy:")
print(hist_mean)
print("Odchylenie standardowe histogramu krawędzi dla autentycznych twarzy:")
print(hist_std)

plt.bar(bins[:-1], average_histogram, width=(bins[1] - bins[0]), color='b', alpha=0.7)
plt.title("'Średni' histogram krawędzi dla autentycznych twarzy, dalszy kontekst, sobel X")
plt.show()

# Zapisanie wartości do pliku
np.savez("histogram_data_far_X.npz", average_histogram=average_histogram, bins=bins, hist_mean=hist_mean, hist_std=hist_std)